# HemMaskNet — Complete Reviewer-Response Experiment Suite

**One self-contained notebook.** No other files needed. Set `ROOT_DIR` in Part 1
(already pointed at your dataset) and run all cells.

| Part | Experiment | Reviewer | Output |
|---|---|---|---|
| 9  | Fully-tuned baselines: EfficientNet-B0, ResNet-50, MobileNet-V3, ViT-B/16, Swin-T, mask-only, HemMaskNet | R2 #2 | `results/table_baselines.csv` |
| 10 | Repeated stratified k-fold CV + bootstrap & Wilson CIs | R2 #3 | `results/cv_summary.json` |
| 11 | Sensitivity: mask errors, missed detections, lighting, reagent, weak agglutination | R2 #5 | `results/table_sensitivity.csv` |
| 12 | Measured params / FLOPs / latency / throughput / memory | R1 #3, R2 #6 | `results/table_profiling.csv` |
| 13 | Classical HOG-SVM baseline | R2 #2 | `results/table_classical.csv` |

---
## ⚠️ Critical bug fixed in this version

Your original `determine_blood_group()` returned `5` (O−) both for AB samples and for
images with fewer than three drops. The paper says these are **excluded, not
relabelled**. Here they return `None` and are dropped, and **Part 15.1 prints the exact
exclusion counts** for Section III-B.

---
# Part 0 — Install & imports

In [ ]:
# !pip install torch torchvision optuna thop scikit-learn scikit-image \
#              opencv-python pandas matplotlib seaborn tqdm scipy Pillow --quiet

In [1]:
from __future__ import annotations
import os, json, math, random, time, copy, io
from collections import Counter
from dataclasses import dataclass, field
from typing import Callable, Dict, List, Optional, Sequence, Tuple

import numpy as np, pandas as pd
from PIL import Image, ImageDraw, ImageFilter, ImageEnhance

import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models

from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix, precision_recall_fscore_support)
from sklearn.model_selection import StratifiedKFold, train_test_split

import matplotlib.pyplot as plt, seaborn as sns
from tqdm.auto import tqdm
import warnings; warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RESULTS_DIR = "results"; os.makedirs(RESULTS_DIR, exist_ok=True)
print("torch :", torch.__version__)
print("device:", DEVICE)
if DEVICE == "cuda":
    print("gpu   :", torch.cuda.get_device_name(0))
    print("vram  : %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))

torch : 2.0.1+cu118
device: cpu


/home/fawadsalamkhan/miniconda3/envs/rl_gpu/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from __future__ import annotations

import os
import json
import math
import random
import time
from collections import Counter
from dataclasses import dataclass
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageEnhance

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

---
# Part 1 — Configuration

**`ROOT_DIR` is already set to your dataset.** If your data is not split into
train/val/test folders, the loader creates a fixed, stratified 80/10/10 split.

In [5]:
# ======================================================================
#  >>> YOUR DATASET ROOT <<<
#  The auto-detector below handles any of these layouts:
#    - ROOT/{train,val,test}/{images,labels}/
#    - ROOT/{train,val,test}/   (images + .txt together)
#    - ROOT/{images,labels}/    (single pool -> auto 80/10/10 split)
#    - ROOT/   (images + .txt flat -> auto 80/10/10 split)
#  Folder names are case-insensitive; synonyms (validation, anns, ...) accepted.
# ======================================================================
ROOT_DIR = "/home/fawadsalamkhan/MyProjects/BloodGroup"

BLOOD_GROUPS = ["A+", "A-", "B+", "B-", "O+", "O-"]
NUM_CLASSES = 6
IMG_SIZE = 224
SEED = 42

DROP_INTENSITY = {0: 200, 1: 200, 2: 150, 3: 150, 4: 255, 5: 255}
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

---
# Part 2 — Label derivation (the bug fix)

`determine_blood_group()` returns `None` (= exclude) for AB phenotypes and for
incompletely-annotated samples, instead of silently calling them O−.

In [6]:
def determine_blood_group(drop_classes: Sequence[int]) -> Optional[int]:
    """Three ordered drop annotations -> blood-group index, or None (=exclude).

    Returns None when the sample is unusable or out of scope:
      * fewer than three drops annotated -> invalid
      * BOTH Anti-A and Anti-B agglutinate -> AB phenotype, OUT OF SCOPE

    The original notebook returned 5 (O-) in both cases -- a silent, clinically
    dangerous mislabel. This function makes the code match the paper.
    """
    if drop_classes is None or len(drop_classes) < 3:
        return None
    clot_A, clot_B, clot_Rh = drop_classes[0], drop_classes[1], drop_classes[2]
    is_A_positive = (clot_A == 0)
    is_B_positive = (clot_B == 2)
    is_Rh_positive = (clot_Rh == 4)
    if is_A_positive and is_B_positive:
        return None                                # AB -> excluded
    if is_A_positive and not is_B_positive:
        return 0 if is_Rh_positive else 1          # A+ / A-
    if is_B_positive and not is_A_positive:
        return 2 if is_Rh_positive else 3          # B+ / B-
    return 4 if is_Rh_positive else 5              # O+ / O-


def parse_annotation(ann_path: str) -> List[Tuple[int, float, float, float, float]]:
    """Parse a YOLO .txt -> up to 3 drops sorted left-to-right (Anti-A, -B, -D)."""
    if not os.path.exists(ann_path):
        return []
    rows = []
    with open(ann_path, "r") as fh:
        for line in fh:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            try:
                cid = int(float(parts[0]))
                xc, yc, w, h = (float(parts[1]), float(parts[2]),
                                float(parts[3]), float(parts[4]))
            except ValueError:
                continue
            rows.append((cid, xc, yc, w, h))
    rows.sort(key=lambda r: r[1])
    return rows[:3]


@dataclass
class Sample:
    img_path: str
    ann_path: str
    drops: List[Tuple[int, float, float, float, float]]
    label: int
    split: str


# ---- flexible dataset-layout discovery -----------------------------------

_IMG_EXT = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff")


def _find_images_labels_pair(directory: str) -> Optional[Tuple[str, str]]:
    """Given a directory, return (images_dir, labels_dir) if discoverable."""
    if not os.path.isdir(directory):
        return None
    entries = {e.lower(): e for e in os.listdir(directory)}
    img_dir = lbl_dir = None
    for cand in ("images", "image", "imgs", "img"):
        if cand in entries:
            img_dir = os.path.join(directory, entries[cand])
            break
    for cand in ("labels", "label", "annotations", "anns", "txt"):
        if cand in entries:
            lbl_dir = os.path.join(directory, entries[cand])
            break
    if img_dir and lbl_dir:
        return img_dir, lbl_dir
    # flat: images and .txt sit directly in `directory`
    has_imgs = any(f.lower().endswith(_IMG_EXT) for f in os.listdir(directory))
    has_txt = any(f.lower().endswith(".txt") for f in os.listdir(directory))
    if has_imgs and has_txt:
        return directory, directory
    return None


def discover_layout(root_dir: str, verbose: bool = True) -> Dict[str, Tuple[str, str]]:
    """Figure out where images and labels live.

    Handles:
      A) root/{train,val,test}/{images,labels}/     (canonical)
      B) root/{train,val,test}/  with flat images+txt
      C) root/{images,labels}/                       (single pool, no splits)
      D) root/  flat images+txt                       (single pool, no splits)
      + case-insensitive folder names and a few synonyms.

    Returns {split_name: (images_dir, labels_dir)}.
    """
    if not os.path.isdir(root_dir):
        raise FileNotFoundError(f"ROOT_DIR does not exist: {root_dir}")

    layout: Dict[str, Tuple[str, str]] = {}
    entries = {e.lower(): e for e in os.listdir(root_dir)}

    split_syn = {"train": ["train", "training"],
                 "val": ["val", "valid", "validation", "dev"],
                 "test": ["test", "testing", "eval"]}

    for split, syns in split_syn.items():
        for s in syns:
            if s in entries:
                pair = _find_images_labels_pair(os.path.join(root_dir, entries[s]))
                if pair:
                    layout[split] = pair
                    break

    if not layout:                                   # no splits -> single pool
        pair = _find_images_labels_pair(root_dir)
        if pair:
            layout["all"] = pair

    if verbose:
        if layout:
            print("Detected dataset layout:")
            for k, (i, l) in layout.items():
                ni = len([f for f in os.listdir(i) if f.lower().endswith(_IMG_EXT)])
                print(f"  [{k:5s}] images={i}  ({ni} files)")
                print(f"          labels={l}")
        else:
            print("Could NOT detect a usable layout under:", root_dir)
            print("Top-level contents:", sorted(os.listdir(root_dir))[:40])
    return layout


def _scan_pair(img_dir: str, lbl_dir: str, split: str, verbose: bool = True):
    files = sorted(f for f in os.listdir(img_dir) if f.lower().endswith(_IMG_EXT))
    samples: List[Sample] = []
    stats = {"total": len(files), "kept": 0, "excluded_ab": 0, "excluded_invalid": 0}
    for fname in files:
        ann_path = os.path.join(lbl_dir, os.path.splitext(fname)[0] + ".txt")
        drops = parse_annotation(ann_path)
        drop_ids = [d[0] for d in drops]
        label = determine_blood_group(drop_ids)
        if label is None:
            if len(drop_ids) >= 3 and drop_ids[0] == 0 and drop_ids[1] == 2:
                stats["excluded_ab"] += 1
            else:
                stats["excluded_invalid"] += 1
            continue
        samples.append(Sample(os.path.join(img_dir, fname), ann_path, drops, label, split))
        stats["kept"] += 1
    if verbose:
        print(f"[{split:5s}] total={stats['total']:4d}  kept={stats['kept']:4d}  "
              f"excluded_AB={stats['excluded_ab']:3d}  excluded_invalid={stats['excluded_invalid']:3d}")
    return samples, stats


def load_dataset(root_dir: str, val_frac: float = 0.10, test_frac: float = 0.10,
                 verbose: bool = True):
    """Load the dataset regardless of layout.

    Returns (train_s, val_s, test_s, stats_by_split).
    If the data has no train/val/test folders, it is split here with a fixed
    stratified seed so results stay reproducible.
    """
    from sklearn.model_selection import train_test_split
    layout = discover_layout(root_dir, verbose=verbose)
    if not layout:
        raise FileNotFoundError(
            f"No images+labels found under {root_dir}. "
            "Expected either {train,val,test}/{images,labels}/ or a flat images+.txt folder.")

    stats_by_split = {}

    if {"train", "val", "test"}.issubset(layout.keys()):
        tr, s1 = _scan_pair(*layout["train"], "train", verbose)
        va, s2 = _scan_pair(*layout["val"], "val", verbose)
        te, s3 = _scan_pair(*layout["test"], "test", verbose)
        stats_by_split = {"train": s1, "val": s2, "test": s3}
        return tr, va, te, stats_by_split

    # partial splits or single pool -> gather everything, then split ourselves
    pool: List[Sample] = []
    agg = {"total": 0, "kept": 0, "excluded_ab": 0, "excluded_invalid": 0}
    for k, (i, l) in layout.items():
        s, st = _scan_pair(i, l, k, verbose)
        pool += s
        for key in agg:
            agg[key] += st[key]

    if verbose:
        print(f"\nNo explicit train/val/test split found -> creating a stratified "
              f"{int(100*(1-val_frac-test_frac))}/{int(100*val_frac)}/{int(100*test_frac)} "
              f"split (seed={SEED}).")
    y = [s.label for s in pool]
    tr, tmp = train_test_split(pool, test_size=val_frac + test_frac,
                               stratify=y, random_state=SEED)
    rel = test_frac / (val_frac + test_frac)
    va, te = train_test_split(tmp, test_size=rel,
                              stratify=[s.label for s in tmp], random_state=SEED)
    for s in tr: s.split = "train"
    for s in va: s.split = "val"
    for s in te: s.split = "test"
    stats_by_split = {"pool": agg}
    if verbose:
        print(f"  -> train={len(tr)}  val={len(va)}  test={len(te)}")
    return tr, va, te, stats_by_split


# backward-compatible helper (canonical layout only)
def scan_split(root_dir: str, split: str, verbose: bool = True):
    pair = _find_images_labels_pair(os.path.join(root_dir, split))
    if pair is None:
        raise FileNotFoundError(f"Could not find images/labels for split '{split}' under {root_dir}")
    return _scan_pair(pair[0], pair[1], split, verbose)

In [7]:
@dataclass
class MaskPerturbation:
    shift_px: int = 0
    scale: float = 1.0
    drop_prob: float = 0.0
    swap_prob: float = 0.0
    mask_off: bool = False

    def is_identity(self):
        return (self.shift_px == 0 and self.scale == 1.0 and self.drop_prob == 0.0
                and self.swap_prob == 0.0 and not self.mask_off)

    def tag(self):
        if self.mask_off: return "mask_off"
        if self.is_identity(): return "clean"
        bits = []
        if self.shift_px: bits.append(f"shift{self.shift_px}")
        if self.scale != 1.0: bits.append(f"scale{self.scale:g}")
        if self.drop_prob: bits.append(f"miss{self.drop_prob:g}")
        if self.swap_prob: bits.append(f"swap{self.swap_prob:g}")
        return "+".join(bits)


@dataclass
class Photometric:
    brightness: float = 1.0
    contrast: float = 1.0
    saturation: float = 1.0
    hue_shift: float = 0.0
    blur_radius: float = 0.0
    noise_std: float = 0.0
    jpeg_quality: int = 0

    def is_identity(self):
        return (self.brightness == 1.0 and self.contrast == 1.0 and self.saturation == 1.0
                and self.hue_shift == 0.0 and self.blur_radius == 0.0
                and self.noise_std == 0.0 and self.jpeg_quality == 0)

    def tag(self):
        if self.is_identity(): return "clean"
        bits = []
        for k, v, d in [("bright", self.brightness, 1.0), ("contr", self.contrast, 1.0),
                        ("sat", self.saturation, 1.0), ("hue", self.hue_shift, 0.0),
                        ("blur", self.blur_radius, 0.0), ("noise", self.noise_std, 0.0),
                        ("jpeg", self.jpeg_quality, 0)]:
            if v != d: bits.append(f"{k}{v:g}")
        return "+".join(bits)

    def apply(self, img, rng):
        import io
        if self.brightness != 1.0: img = ImageEnhance.Brightness(img).enhance(self.brightness)
        if self.contrast != 1.0:   img = ImageEnhance.Contrast(img).enhance(self.contrast)
        if self.saturation != 1.0: img = ImageEnhance.Color(img).enhance(self.saturation)
        if self.hue_shift != 0.0:
            hsv = np.array(img.convert("HSV"), dtype=np.int16)
            hsv[..., 0] = (hsv[..., 0] + int(self.hue_shift * 255)) % 256
            img = Image.fromarray(hsv.astype(np.uint8), mode="HSV").convert("RGB")
        if self.blur_radius > 0: img = img.filter(ImageFilter.GaussianBlur(self.blur_radius))
        if self.jpeg_quality:
            buf = io.BytesIO(); img.save(buf, format="JPEG", quality=int(self.jpeg_quality))
            buf.seek(0); img = Image.open(buf).convert("RGB")
        if self.noise_std > 0:
            arr = np.asarray(img, dtype=np.float32) / 255.0
            arr = arr + np.random.normal(0.0, self.noise_std, arr.shape)
            img = Image.fromarray((np.clip(arr, 0, 1) * 255).astype(np.uint8))
        return img

---
# Part 4 — Dataset + flexible layout discovery

`discover_layout()` and `load_dataset()` find your images/labels whatever the folder
structure, and split automatically if there are no train/val/test folders.

In [8]:
class BloodTypingDataset(Dataset):
    def __init__(self, samples, img_size=IMG_SIZE, train_aug=False,
                 mask_perturb=None, photometric=None, perturb_seed=SEED):
        self.samples = samples
        self.labels = [s.label for s in samples]
        self.img_size = img_size
        self.mask_perturb = mask_perturb or MaskPerturbation()
        self.photometric = photometric or Photometric()
        self.perturb_seed = perturb_seed
        aug = [transforms.ColorJitter(0.2, 0.2, 0.1)] if train_aug else []
        self.img_tf = transforms.Compose(
            [transforms.Resize((img_size, img_size))] + aug +
            [transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
        self.mask_tf = transforms.Compose(
            [transforms.Resize((img_size, img_size)), transforms.ToTensor()])

    def __len__(self):
        return len(self.samples)

    def _render_mask(self, s, W, H, rng):
        mask = Image.new("L", (W, H), 0)
        if self.mask_perturb.mask_off:
            return mask
        draw = ImageDraw.Draw(mask)
        drops = list(s.drops)
        if self.mask_perturb.swap_prob > 0 and len(drops) >= 2 and rng.random() < self.mask_perturb.swap_prob:
            i, j = rng.sample(range(len(drops)), 2)
            ci, cj = drops[i][0], drops[j][0]
            drops[i] = (cj,) + tuple(drops[i][1:])
            drops[j] = (ci,) + tuple(drops[j][1:])
        for cid, xc, yc, w, h in drops:
            if self.mask_perturb.drop_prob > 0 and rng.random() < self.mask_perturb.drop_prob:
                continue
            cx, cy = xc * W, yc * H
            bw, bh = w * W * self.mask_perturb.scale, h * H * self.mask_perturb.scale
            if self.mask_perturb.shift_px:
                p = self.mask_perturb.shift_px
                cx += rng.uniform(-p, p); cy += rng.uniform(-p, p)
            x1, y1 = int(cx - bw / 2), int(cy - bh / 2)
            x2, y2 = int(cx + bw / 2), int(cy + bh / 2)
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(W - 1, x2), min(H - 1, y2)
            if x2 > x1 and y2 > y1:
                draw.ellipse([x1, y1, x2, y2], fill=DROP_INTENSITY.get(cid, 255))
        return mask

    def __getitem__(self, idx):
        s = self.samples[idx]
        rng = random.Random(self.perturb_seed * 1_000_003 + idx)
        try:
            img = Image.open(s.img_path).convert("RGB")
        except Exception:
            img = Image.new("RGB", (self.img_size, self.img_size), (128, 128, 128))
        W, H = img.size
        mask = self._render_mask(s, W, H, rng)
        if not self.photometric.is_identity():
            img = self.photometric.apply(img, rng)
        return self.img_tf(img), self.mask_tf(mask), torch.tensor(s.label, dtype=torch.long)


def make_weighted_sampler(labels):
    counts = Counter(labels); n = len(labels)
    w = {c: n / cnt for c, cnt in counts.items() if cnt > 0}
    weights = torch.tensor([w.get(l, 0.0) for l in labels], dtype=torch.double)
    return WeightedRandomSampler(weights, num_samples=n, replacement=True)

In [9]:
BACKBONES = ["efficientnet_b0", "resnet50", "mobilenet_v3_large", "vit_b_16", "swin_t"]


def build_backbone(name, pretrained=True):
    w = "DEFAULT" if pretrained else None
    if name == "efficientnet_b0":
        m = models.efficientnet_b0(weights=w); dim = m.classifier[1].in_features; m.classifier = nn.Identity()
    elif name == "resnet50":
        m = models.resnet50(weights=w); dim = m.fc.in_features; m.fc = nn.Identity()
    elif name == "mobilenet_v3_large":
        m = models.mobilenet_v3_large(weights=w); dim = m.classifier[0].in_features; m.classifier = nn.Identity()
    elif name == "vit_b_16":
        m = models.vit_b_16(weights=w); dim = m.heads.head.in_features; m.heads = nn.Identity()
    elif name == "swin_t":
        m = models.swin_t(weights=w); dim = m.head.in_features; m.head = nn.Identity()
    else:
        raise ValueError(f"Unknown backbone: {name}")
    return m, dim


class MaskEncoder(nn.Module):
    def __init__(self, out_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, out_dim, 3, padding=1), nn.BatchNorm2d(out_dim), nn.ReLU(), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((1, 1)))
        self.out_dim = out_dim

    def forward(self, m):
        return self.net(m).flatten(1)


class DualStreamNet(nn.Module):
    def __init__(self, backbone="efficientnet_b0", use_image=True, use_mask=True,
                 num_classes=NUM_CLASSES, hidden1=256, hidden2=128,
                 dropout1=0.5, dropout2=0.3, pretrained=True, mask_dim=128):
        super().__init__()
        assert use_image or use_mask
        self.use_image, self.use_mask, self.backbone_name = use_image, use_mask, backbone
        dim = 0
        if use_image:
            self.backbone, img_dim = build_backbone(backbone, pretrained); dim += img_dim
        if use_mask:
            self.mask_encoder = MaskEncoder(mask_dim); dim += mask_dim
        self.fused_dim = dim
        self.classifier = nn.Sequential(
            nn.Linear(dim, hidden1), nn.BatchNorm1d(hidden1), nn.ReLU(), nn.Dropout(dropout1),
            nn.Linear(hidden1, hidden2), nn.BatchNorm1d(hidden2), nn.ReLU(), nn.Dropout(dropout2),
            nn.Linear(hidden2, num_classes))

    def forward(self, x, mask):
        feats = []
        if self.use_image: feats.append(self.backbone(x).flatten(1))
        if self.use_mask:  feats.append(self.mask_encoder(mask))
        return self.classifier(torch.cat(feats, dim=1) if len(feats) > 1 else feats[0])


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

---
# Part 6 — Training & evaluation (one shared loop)

In [10]:
@dataclass
class HParams:
    learning_rate: float = 5.699e-4
    batch_size: int = 8
    dropout1: float = 0.5
    dropout2: float = 0.319
    weight_decay: float = 8.545e-4
    hidden1: int = 512
    hidden2: int = 64
    optimizer: str = "AdamW"
    scheduler_patience: int = 7
    epochs: int = 30

    @staticmethod
    def from_optuna(d, epochs=30):
        return HParams(d["learning_rate"], d["batch_size"], d["dropout_rate1"], d["dropout_rate2"],
                       d["weight_decay"], d["hidden_size1"], d["hidden_size2"],
                       d["optimizer"], d["scheduler_patience"], epochs)


def build_optimizer(name, params, lr, wd):
    if name == "Adam": return torch.optim.Adam(params, lr=lr, weight_decay=wd)
    if name == "AdamW": return torch.optim.AdamW(params, lr=lr, weight_decay=wd)
    return torch.optim.SGD(params, lr=lr, momentum=0.9, weight_decay=wd)


@torch.no_grad()
def evaluate(model, loader, device, criterion=None):
    model.eval()
    y_true, y_pred, y_prob = [], [], []
    total_loss, n = 0.0, 0
    for imgs, masks, labels in loader:
        imgs, masks, labels = imgs.to(device), masks.to(device), labels.to(device)
        logits = model(imgs, masks)
        if criterion is not None:
            total_loss += criterion(logits, labels).item() * labels.size(0)
        n += labels.size(0)
        prob = torch.softmax(logits, dim=1)
        y_prob.append(prob.cpu().numpy()); y_pred.append(prob.argmax(1).cpu().numpy())
        y_true.append(labels.cpu().numpy())
    y_true = np.concatenate(y_true); y_pred = np.concatenate(y_pred); y_prob = np.concatenate(y_prob)
    return {"y_true": y_true, "y_pred": y_pred, "y_prob": y_prob,
            "accuracy": float(accuracy_score(y_true, y_pred)),
            "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
            "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
            "loss": (total_loss / n) if criterion is not None and n else float("nan")}


def train_model(model, train_samples, val_samples, hp, device,
                trial=None, verbose=False, train_aug=False):
    train_ds = BloodTypingDataset(train_samples, train_aug=train_aug)
    val_ds = BloodTypingDataset(val_samples)
    train_loader = DataLoader(train_ds, batch_size=hp.batch_size,
                              sampler=make_weighted_sampler(train_ds.labels), num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=hp.batch_size, shuffle=False, num_workers=2)
    model.to(device)
    opt = build_optimizer(hp.optimizer, model.parameters(), hp.learning_rate, hp.weight_decay)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min",
                                                       patience=hp.scheduler_patience, factor=0.5)
    criterion = nn.CrossEntropyLoss()
    hist = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_val_acc, best_state = 0.0, None
    for epoch in range(hp.epochs):
        model.train()
        run_loss, correct, total = 0.0, 0, 0
        for imgs, masks, labels in train_loader:
            imgs, masks, labels = imgs.to(device), masks.to(device), labels.to(device)
            opt.zero_grad()
            logits = model(imgs, masks)
            loss = criterion(logits, labels)
            loss.backward(); opt.step()
            run_loss += loss.item() * labels.size(0)
            correct += (logits.argmax(1) == labels).sum().item()
            total += labels.size(0)
        hist["train_loss"].append(run_loss / max(total, 1))
        hist["train_acc"].append(correct / max(total, 1))
        vres = evaluate(model, val_loader, device, criterion)
        hist["val_loss"].append(vres["loss"]); hist["val_acc"].append(vres["accuracy"])
        sched.step(vres["loss"])
        if vres["accuracy"] > best_val_acc:
            best_val_acc = vres["accuracy"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        if verbose:
            print(f"  ep{epoch+1:02d}  train_loss={hist['train_loss'][-1]:.4f} "
                  f"acc={hist['train_acc'][-1]:.3f} | val_loss={vres['loss']:.4f} "
                  f"acc={vres['accuracy']:.3f}")
        if trial is not None:
            trial.report(vres["accuracy"], epoch)
            import optuna
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()
    if best_state is not None:
        model.load_state_dict(best_state)
    hist["best_val_acc"] = best_val_acc
    return model, hist

In [11]:
def wilson_ci(correct, n, confidence=0.95):
    if n == 0: return (float("nan"), float("nan"))
    from scipy.stats import norm
    z = norm.ppf(1 - (1 - confidence) / 2); p = correct / n
    denom = 1 + z**2 / n; centre = p + z**2 / (2 * n)
    half = z * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2))
    return (max(0.0, (centre - half) / denom), min(1.0, (centre + half) / denom))


def bootstrap_ci(y_true, y_pred, metric="accuracy", n_boot=10000, confidence=0.95, seed=SEED):
    rng = np.random.default_rng(seed); n = len(y_true); stats = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, n); yt, yp = y_true[idx], y_pred[idx]
        stats[b] = (yt == yp).mean() if metric == "accuracy" else \
            f1_score(yt, yp, average="macro", zero_division=0)
    alpha = (1 - confidence) / 2
    point = ((y_true == y_pred).mean() if metric == "accuracy"
             else f1_score(y_true, y_pred, average="macro", zero_division=0))
    return {"point": float(point), "lo": float(np.quantile(stats, alpha)),
            "hi": float(np.quantile(stats, 1 - alpha)), "std": float(stats.std()), "n_boot": n_boot}


def expected_calibration_error(y_true, y_prob, n_bins=10):
    conf = y_prob.max(axis=1); pred = y_prob.argmax(axis=1)
    acc = (pred == y_true).astype(float); bins = np.linspace(0, 1, n_bins + 1)
    ece, n = 0.0, len(y_true)
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.sum() == 0: continue
        ece += (m.sum() / n) * abs(acc[m].mean() - conf[m].mean())
    return float(ece)


def per_class_table(y_true, y_pred):
    import pandas as pd
    rep = classification_report(y_true, y_pred, labels=list(range(NUM_CLASSES)),
                                target_names=BLOOD_GROUPS, output_dict=True, zero_division=0)
    rows = []
    for g in BLOOD_GROUPS:
        r = rep[g]; n = int(r["support"]); correct = int(round(r["recall"] * n))
        lo, hi = wilson_ci(correct, n) if n else (float("nan"),) * 2
        rows.append({"Class": g, "N": n, "Correct": correct,
                     "Precision": r["precision"], "Recall": r["recall"], "F1": r["f1-score"],
                     "Recall 95% CI": f"[{lo:.2f}, {hi:.2f}]" if n else "-"})
    return pd.DataFrame(rows)

### Self-test — verify label logic + CI before touching real data

In [12]:
_cases = {(0,3,4):"A+",(0,3,5):"A-",(1,2,4):"B+",(1,2,5):"B-",(1,3,4):"O+",
          (1,3,5):"O-",(0,2,4):"EXCLUDED (AB+)",(0,2,5):"EXCLUDED (AB-)"}
print("Label derivation self-test")
_ok = True
for _d,_exp in _cases.items():
    _r = determine_blood_group(list(_d))
    _got = BLOOD_GROUPS[_r] if _r is not None else "EXCLUDED"
    _pass = (_got == _exp) or (_r is None and _exp.startswith("EXCLUDED"))
    _ok &= _pass
    print(f"  {'OK ' if _pass else 'FAIL'} drops={_d} -> {_got:9s} (expected {_exp})")
print(f"  {'OK ' if determine_blood_group([0,3]) is None else 'FAIL'} incomplete -> EXCLUDED")
_lo,_hi = wilson_ci(24,24)
print(f"\nWilson CI 24/24 = [{100*_lo:.1f}%, {100*_hi:.1f}%]  (paper: ~[86.2%, 100%])")
assert _ok, "label logic self-test FAILED"
print("\nAll checks passed.")

Label derivation self-test
  OK  drops=(0, 3, 4) -> A+        (expected A+)
  OK  drops=(0, 3, 5) -> A-        (expected A-)
  OK  drops=(1, 2, 4) -> B+        (expected B+)
  OK  drops=(1, 2, 5) -> B-        (expected B-)
  OK  drops=(1, 3, 4) -> O+        (expected O+)
  OK  drops=(1, 3, 5) -> O-        (expected O-)
  OK  drops=(0, 2, 4) -> EXCLUDED  (expected EXCLUDED (AB+))
  OK  drops=(0, 2, 5) -> EXCLUDED  (expected EXCLUDED (AB-))
  OK  incomplete -> EXCLUDED

Wilson CI 24/24 = [86.2%, 100.0%]  (paper: ~[86.2%, 100%])

All checks passed.


In [13]:
from __future__ import annotations

import copy
import json
import os
import time
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)

---
# Part 8 — Shared Optuna tuner (identical budget for every model)

In [14]:
def tune(backbone, use_image, use_mask, train_samples, val_samples,
         n_trials=30, epochs=20, seed=SEED):
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    def objective(trial):
        hp = HParams(
            learning_rate=trial.suggest_float("learning_rate", 1e-5, 1e-3, log=True),
            batch_size=trial.suggest_categorical("batch_size", [8, 16, 32]),
            dropout1=trial.suggest_float("dropout_rate1", 0.2, 0.7),
            dropout2=trial.suggest_float("dropout_rate2", 0.1, 0.5),
            weight_decay=trial.suggest_float("weight_decay", 1e-5, 1e-2, log=True),
            hidden1=trial.suggest_categorical("hidden_size1", [128, 256, 512]),
            hidden2=trial.suggest_categorical("hidden_size2", [64, 128, 256]),
            optimizer=trial.suggest_categorical("optimizer", ["Adam", "AdamW", "SGD"]),
            scheduler_patience=trial.suggest_int("scheduler_patience", 3, 10),
            epochs=epochs)
        set_seed(seed)
        model = DualStreamNet(backbone=backbone, use_image=use_image, use_mask=use_mask,
                              hidden1=hp.hidden1, hidden2=hp.hidden2,
                              dropout1=hp.dropout1, dropout2=hp.dropout2)
        _, hist = train_model(model, train_samples, val_samples, hp, DEVICE, trial=trial)
        del model
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        return hist["best_val_acc"]

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=seed),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10, interval_steps=1))
    study.optimize(objective, n_trials=n_trials, gc_after_trial=True)
    return study.best_params

---
# Part 9 — Experiment 1: fully-tuned baselines *(R2 #2)*

In [15]:
MODEL_GRID = [
    ("EfficientNet-B0 (image only)", "efficientnet_b0",    True,  False),
    ("ResNet-50 (image only)",       "resnet50",           True,  False),
    ("MobileNet-V3-L (image only)",  "mobilenet_v3_large", True,  False),
    ("ViT-B/16 (image only)",        "vit_b_16",           True,  False),
    ("Swin-T (image only)",          "swin_t",             True,  False),
    ("Mask encoder only",            "efficientnet_b0",    False, True),
    ("HemMaskNet (dual, ours)",      "efficientnet_b0",    True,  True),
]


def exp1_baselines(train_samples, val_samples, test_samples,
                   n_trials=30, tune_epochs=20, final_epochs=30, n_seeds=3,
                   out_csv=f"{RESULTS_DIR}/table_baselines.csv"):
    rows = []
    for name, backbone, use_img, use_mask in MODEL_GRID:
        print(f"\n=== {name} : tuning ({n_trials} trials) ===")
        best = tune(backbone, use_img, use_mask, train_samples, val_samples,
                    n_trials=n_trials, epochs=tune_epochs)
        hp = HParams.from_optuna(best, epochs=final_epochs)
        print(f"    best params: {best}")
        accs, f1s, val_accs = [], [], []
        last_eval = None
        for s in range(n_seeds):
            set_seed(SEED + s)
            model = DualStreamNet(backbone=backbone, use_image=use_img, use_mask=use_mask,
                                  hidden1=hp.hidden1, hidden2=hp.hidden2,
                                  dropout1=hp.dropout1, dropout2=hp.dropout2)
            model, hist = train_model(model, train_samples, val_samples, hp, DEVICE)
            test_loader = DataLoader(BloodTypingDataset(test_samples),
                                     batch_size=hp.batch_size, shuffle=False)
            res = evaluate(model, test_loader, DEVICE)
            accs.append(res["accuracy"]); f1s.append(res["macro_f1"]); val_accs.append(hist["best_val_acc"])
            last_eval = res
            print(f"    seed {s}: test acc={res['accuracy']:.4f}  macroF1={res['macro_f1']:.4f}")
            del model
            if torch.cuda.is_available(): torch.cuda.empty_cache()
        boot = bootstrap_ci(last_eval["y_true"], last_eval["y_pred"], "accuracy")
        n = len(last_eval["y_true"]); correct = int(last_eval["accuracy"] * n)
        wlo, whi = wilson_ci(correct, n)
        rows.append({
            "Model": name,
            "Params (M)": round(count_parameters(
                DualStreamNet(backbone=backbone, use_image=use_img, use_mask=use_mask,
                              hidden1=hp.hidden1, hidden2=hp.hidden2)) / 1e6, 2),
            "Val Acc (%)": f"{100*np.mean(val_accs):.1f}",
            "Test Acc (%)": f"{100*np.mean(accs):.1f} ± {100*np.std(accs):.1f}",
            "Macro F1": f"{np.mean(f1s):.3f} ± {np.std(f1s):.3f}",
            "Wilson 95% CI": f"[{100*wlo:.1f}, {100*whi:.1f}]",
            "Bootstrap 95% CI": f"[{100*boot['lo']:.1f}, {100*boot['hi']:.1f}]",
            "best_params": json.dumps(best)})
        pd.DataFrame(rows).to_csv(out_csv, index=False)
    df = pd.DataFrame(rows); df.to_csv(out_csv, index=False)
    print(f"\nSaved -> {out_csv}")
    return df

---
# Part 10 — Experiment 2: cross-validation *(R2 #3)*

In [16]:
def exp2_cross_validation(all_samples, hp, backbone="efficientnet_b0",
                          use_image=True, use_mask=True, n_splits=5, n_repeats=2,
                          out_prefix=f"{RESULTS_DIR}/cv"):
    y = np.array([s.label for s in all_samples])
    fold_rows = []
    oof_pred = np.full(len(all_samples), -1, dtype=int)
    oof_prob = np.zeros((len(all_samples), NUM_CLASSES), dtype=float)
    for rep in range(n_repeats):
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED + rep)
        for fold, (tr_idx, te_idx) in enumerate(skf.split(np.zeros(len(y)), y)):
            tr_y = y[tr_idx]
            inner_tr, inner_val = train_test_split(tr_idx, test_size=0.12,
                                                   stratify=tr_y, random_state=SEED + rep)
            tr_s = [all_samples[i] for i in inner_tr]
            va_s = [all_samples[i] for i in inner_val]
            te_s = [all_samples[i] for i in te_idx]
            set_seed(SEED + rep * 100 + fold)
            model = DualStreamNet(backbone=backbone, use_image=use_image, use_mask=use_mask,
                                  hidden1=hp.hidden1, hidden2=hp.hidden2,
                                  dropout1=hp.dropout1, dropout2=hp.dropout2)
            model, _ = train_model(model, tr_s, va_s, hp, DEVICE)
            loader = DataLoader(BloodTypingDataset(te_s), batch_size=hp.batch_size, shuffle=False)
            res = evaluate(model, loader, DEVICE)
            if rep == 0:
                oof_pred[te_idx] = res["y_pred"]; oof_prob[te_idx] = res["y_prob"]
            fold_rows.append({"Repeat": rep + 1, "Fold": fold + 1, "N test": len(te_idx),
                              "Accuracy": res["accuracy"], "Macro F1": res["macro_f1"]})
            print(f"  rep{rep+1} fold{fold+1}: n={len(te_idx):3d} "
                  f"acc={res['accuracy']:.4f} f1={res['macro_f1']:.4f}")
            del model
            if torch.cuda.is_available(): torch.cuda.empty_cache()
    df = pd.DataFrame(fold_rows); df.to_csv(f"{out_prefix}_folds.csv", index=False)
    mask = oof_pred >= 0; yt, yp = y[mask], oof_pred[mask]
    acc_boot = bootstrap_ci(yt, yp, "accuracy"); f1_boot = bootstrap_ci(yt, yp, "macro_f1")
    wlo, whi = wilson_ci(int((yt == yp).sum()), len(yt))
    summary = {
        "n_samples_oof": int(len(yt)),
        "fold_acc_mean": float(df["Accuracy"].mean()), "fold_acc_std": float(df["Accuracy"].std()),
        "fold_f1_mean": float(df["Macro F1"].mean()), "fold_f1_std": float(df["Macro F1"].std()),
        "oof_accuracy": float(accuracy_score(yt, yp)),
        "oof_accuracy_boot_ci": [acc_boot["lo"], acc_boot["hi"]],
        "oof_accuracy_wilson_ci": [wlo, whi],
        "oof_macro_f1": float(f1_score(yt, yp, average="macro", zero_division=0)),
        "oof_macro_f1_boot_ci": [f1_boot["lo"], f1_boot["hi"]],
        "oof_ece": expected_calibration_error(yt, oof_prob[mask]),
        "oof_confusion_matrix": confusion_matrix(yt, yp, labels=list(range(NUM_CLASSES))).tolist()}
    with open(f"{out_prefix}_summary.json", "w") as fh:
        json.dump(summary, fh, indent=2)
    per_class_table(yt, yp).to_csv(f"{out_prefix}_per_class.csv", index=False)
    print("\n--- Cross-validated summary (report THIS, not the small-test number) ---")
    print(f"  {n_repeats}x{n_splits}-fold accuracy : "
          f"{100*summary['fold_acc_mean']:.2f} ± {100*summary['fold_acc_std']:.2f} %")
    print(f"  OOF accuracy (n={summary['n_samples_oof']}) : {100*summary['oof_accuracy']:.2f} % "
          f"[boot 95% CI {100*acc_boot['lo']:.1f}-{100*acc_boot['hi']:.1f}]")
    print(f"  OOF macro-F1 : {summary['oof_macro_f1']:.3f} "
          f"[boot 95% CI {f1_boot['lo']:.3f}-{f1_boot['hi']:.3f}]")
    print(f"  OOF ECE      : {summary['oof_ece']:.3f}")
    return df, summary

---
# Part 11 — Experiment 3: sensitivity / robustness *(R2 #5)*

In [17]:
MASK_PERTURBATIONS = [
    MaskPerturbation(), MaskPerturbation(shift_px=5), MaskPerturbation(shift_px=10),
    MaskPerturbation(shift_px=20), MaskPerturbation(scale=0.7), MaskPerturbation(scale=1.3),
    MaskPerturbation(drop_prob=0.10), MaskPerturbation(drop_prob=0.33),
    MaskPerturbation(swap_prob=0.20), MaskPerturbation(mask_off=True)]

PHOTOMETRIC_PERTURBATIONS = [
    Photometric(), Photometric(brightness=0.7), Photometric(brightness=1.3),
    Photometric(contrast=0.7), Photometric(contrast=1.3),
    Photometric(saturation=0.6), Photometric(saturation=1.4), Photometric(hue_shift=0.05),
    Photometric(blur_radius=1.5), Photometric(blur_radius=3.0),
    Photometric(noise_std=0.05), Photometric(jpeg_quality=40)]


def exp3_sensitivity(model, test_samples, batch_size=8,
                     out_csv=f"{RESULTS_DIR}/table_sensitivity.csv"):
    rows = []

    def run(tag, kind, mp, ph):
        ds = BloodTypingDataset(test_samples, mask_perturb=mp, photometric=ph)
        loader = DataLoader(ds, batch_size=batch_size, shuffle=False)
        res = evaluate(model, loader, DEVICE)
        n = len(res["y_true"]); correct = int(res["accuracy"] * n)
        lo, hi = wilson_ci(correct, n)
        rows.append({"Perturbation type": kind, "Setting": tag,
                     "Accuracy (%)": round(100 * res["accuracy"], 2),
                     "Macro F1": round(res["macro_f1"], 3),
                     "ECE": round(expected_calibration_error(res["y_true"], res["y_prob"]), 3),
                     "95% CI": f"[{100*lo:.1f}, {100*hi:.1f}]"})
        print(f"  {kind:12s} {tag:14s} acc={100*res['accuracy']:6.2f}%  f1={res['macro_f1']:.3f}")

    print("\n--- Mask-error sensitivity ---")
    for mp in MASK_PERTURBATIONS: run(mp.tag(), "Mask", mp, None)
    print("\n--- Appearance / lighting / reagent sensitivity ---")
    for ph in PHOTOMETRIC_PERTURBATIONS: run(ph.tag(), "Appearance", None, ph)

    df = pd.DataFrame(rows)
    clean = df[df["Setting"] == "clean"]["Accuracy (%)"].iloc[0]
    df["Δ vs clean (pp)"] = (df["Accuracy (%)"] - clean).round(2)
    df.to_csv(out_csv, index=False)
    print(f"\nSaved -> {out_csv}")
    return df


def plot_sensitivity(df, path=f"{RESULTS_DIR}/fig_sensitivity.png"):
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), dpi=200)
    for ax, kind in zip(axes, ["Mask", "Appearance"]):
        sub = df[df["Perturbation type"] == kind]
        ax.bar(sub["Setting"], sub["Accuracy (%)"], color="#4B6EA9", edgecolor="black", linewidth=0.5)
        ax.axhline(sub[sub["Setting"] == "clean"]["Accuracy (%)"].iloc[0],
                   ls="--", c="crimson", lw=1.2, label="clean baseline")
        ax.set_title(f"{kind} perturbation sensitivity", fontsize=11)
        ax.set_ylabel("Test accuracy (%)"); ax.set_ylim(0, 105)
        ax.tick_params(axis="x", rotation=60, labelsize=8); ax.legend(fontsize=8)
    plt.tight_layout(); plt.savefig(path, bbox_inches="tight")
    print(f"Saved -> {path}")

---
# Part 12 — Experiment 4: measured complexity & timing *(R1 #3, R2 #6)*

In [18]:
def exp4_profiling(hp, batch_sizes=(1, 8, 32), n_warmup=10, n_iters=100,
                   out_csv=f"{RESULTS_DIR}/table_profiling.csv"):
    try:
        from thop import profile as thop_profile
        have_thop = True
    except ImportError:
        have_thop = False
        print("[warn] `pip install thop` for FLOP counts; skipping FLOPs column.")
    rows = []
    for name, backbone, use_img, use_mask in MODEL_GRID:
        model = DualStreamNet(backbone=backbone, use_image=use_img, use_mask=use_mask,
                              hidden1=hp.hidden1, hidden2=hp.hidden2).eval()
        params = count_parameters(model)
        gflops = float("nan")
        if have_thop:
            x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE); m = torch.randn(1, 1, IMG_SIZE, IMG_SIZE)
            macs, _ = thop_profile(copy.deepcopy(model), inputs=(x, m), verbose=False)
            gflops = 2 * macs / 1e9
        for device in (["cuda", "cpu"] if torch.cuda.is_available() else ["cpu"]):
            model.to(device)
            for bs in batch_sizes:
                x = torch.randn(bs, 3, IMG_SIZE, IMG_SIZE, device=device)
                m = torch.randn(bs, 1, IMG_SIZE, IMG_SIZE, device=device)
                with torch.no_grad():
                    for _ in range(n_warmup): model(x, m)
                    if device == "cuda":
                        torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
                    times = []
                    for _ in range(n_iters):
                        t0 = time.perf_counter(); model(x, m)
                        if device == "cuda": torch.cuda.synchronize()
                        times.append((time.perf_counter() - t0) * 1000)
                times = np.array(times)
                peak_mb = (torch.cuda.max_memory_allocated() / 1e6) if device == "cuda" else float("nan")
                rows.append({"Model": name, "Device": device.upper(), "Batch": bs,
                             "Params (M)": round(params / 1e6, 2),
                             "GFLOPs (bs=1)": round(gflops, 2) if gflops == gflops else "-",
                             "Latency mean (ms)": round(times.mean(), 2),
                             "Latency p95 (ms)": round(float(np.percentile(times, 95)), 2),
                             "Per-image (ms)": round(times.mean() / bs, 2),
                             "Throughput (img/s)": round(bs * 1000 / times.mean(), 1),
                             "Peak GPU mem (MB)": round(peak_mb, 1) if peak_mb == peak_mb else "-"})
                print(f"  {name:32s} {device:4s} bs={bs:2d}  "
                      f"{times.mean():7.2f} ms  {bs*1000/times.mean():7.1f} img/s")
        del model
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    df = pd.DataFrame(rows); df.to_csv(out_csv, index=False)
    print(f"\nSaved -> {out_csv}")
    return df

---
# Part 13 — Experiment 5: classical HOG-SVM baseline *(R2 #2)*

In [19]:
def _hog_features(sample, size=128):
    import cv2
    from skimage.feature import hog
    img = cv2.imread(sample.img_path)
    if img is None:
        return np.zeros(2000, dtype=np.float32)
    H, W = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    global_hog = hog(cv2.resize(gray, (size, size)), orientations=9,
                     pixels_per_cell=(16, 16), cells_per_block=(2, 2), feature_vector=True)
    drop_feats = []
    for k in range(3):
        if k < len(sample.drops):
            _, xc, yc, w, h = sample.drops[k]
            x1, y1 = max(0, int((xc - w / 2) * W)), max(0, int((yc - h / 2) * H))
            x2, y2 = min(W, int((xc + w / 2) * W)), min(H, int((yc + h / 2) * H))
            roi = gray[y1:y2, x1:x2]
            if roi.size == 0:
                drop_feats.extend([0.0] * 5); continue
            edges = cv2.Canny(roi, 50, 150); lap = cv2.Laplacian(roi, cv2.CV_64F)
            drop_feats.extend([float(roi.mean()), float(roi.std()), float(edges.mean()),
                               float(lap.var()),
                               float(np.percentile(roi, 90) - np.percentile(roi, 10))])
        else:
            drop_feats.extend([0.0] * 5)
    return np.concatenate([global_hog, np.array(drop_feats, dtype=np.float32)]).astype(np.float32)


def exp5_classical(train_samples, test_samples, out_csv=f"{RESULTS_DIR}/table_classical.csv"):
    from sklearn.svm import SVC
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.model_selection import GridSearchCV
    print("Extracting HOG + drop-texture features ...")
    Xtr = np.stack([_hog_features(s) for s in train_samples])
    ytr = np.array([s.label for s in train_samples])
    Xte = np.stack([_hog_features(s) for s in test_samples])
    yte = np.array([s.label for s in test_samples])
    pipe = make_pipeline(StandardScaler(), SVC(class_weight="balanced", probability=True))
    grid = GridSearchCV(pipe, {"svc__C": [0.1, 1, 10, 100],
                               "svc__gamma": ["scale", 1e-3, 1e-4],
                               "svc__kernel": ["rbf", "linear"]},
                        cv=5, scoring="f1_macro", n_jobs=-1)
    grid.fit(Xtr, ytr); yp = grid.predict(Xte)
    acc = accuracy_score(yte, yp); f1 = f1_score(yte, yp, average="macro", zero_division=0)
    boot = bootstrap_ci(yte, yp, "accuracy"); lo, hi = wilson_ci(int((yte == yp).sum()), len(yte))
    df = pd.DataFrame([{"Model": "HOG + drop-edge -> SVM (classical)",
                        "Best params": str(grid.best_params_),
                        "Test Acc (%)": round(100 * acc, 2), "Macro F1": round(f1, 3),
                        "Wilson 95% CI": f"[{100*lo:.1f}, {100*hi:.1f}]",
                        "Bootstrap 95% CI": f"[{100*boot['lo']:.1f}, {100*boot['hi']:.1f}]"}])
    df.to_csv(out_csv, index=False); print(df.to_string(index=False))
    return df

---
# Part 14 — LaTeX export

In [20]:
def to_latex(df, caption, label, path, drop_cols=None):
    d = df.drop(columns=[c for c in (drop_cols or []) if c in df.columns])
    tex = d.to_latex(index=False, escape=True, column_format="l" + "c" * (len(d.columns) - 1))
    body = ("\\begin{table}[!t]\n\\centering\n"
            f"\\caption{{{caption}}}\n\\label{{{label}}}\n"
            "\\resizebox{\\columnwidth}{!}{%\n" + tex + "}\n\\end{table}\n")
    with open(path, "w") as fh: fh.write(body)
    print(f"Saved -> {path}")
    return body

---
---
# ▶ Part 15 — RUN THE EXPERIMENTS

Everything above is definitions. Cells below actually execute.

**Smoke-test first:** leave `SMOKE_TEST = True`, run the whole notebook (~15 min),
confirm it completes, then set `False` for the real run.

In [21]:
SMOKE_TEST = True     # <-- set False for the real, publication-grade run
if SMOKE_TEST:
    N_TRIALS, TUNE_EPOCHS, FINAL_EPOCHS, N_SEEDS = 3, 3, 3, 1
    CV_SPLITS, CV_REPEATS = 3, 1
    print(">>> SMOKE TEST — results NOT publication-grade")
else:
    N_TRIALS, TUNE_EPOCHS, FINAL_EPOCHS, N_SEEDS = 30, 20, 30, 3
    CV_SPLITS, CV_REPEATS = 5, 2
    print(">>> FULL RUN — expect 6-14 h on one GPU")
set_seed(SEED)
print(f"trials={N_TRIALS} tune_ep={TUNE_EPOCHS} final_ep={FINAL_EPOCHS} "
      f"seeds={N_SEEDS} cv={CV_REPEATS}x{CV_SPLITS}")

>>> SMOKE TEST — results NOT publication-grade
trials=3 tune_ep=3 final_ep=3 seeds=1 cv=1x3


## 15.1 Load data — auto-detect layout & report exclusions

**The exclusion counts printed here go into Section III-B of the paper.**

In [22]:
assert os.path.isdir(ROOT_DIR), (
    f"ROOT_DIR not found: {ROOT_DIR}\nEdit the ROOT_DIR line in Part 1.")

train_s, val_s, test_s, stats = load_dataset(ROOT_DIR, val_frac=0.10, test_frac=0.10)
all_s = train_s + val_s + test_s

if "train" in stats:      # explicit splits existed
    excl = pd.DataFrame([stats["train"], stats["val"], stats["test"]],
                        index=["train", "val", "test"])
else:                     # single pool was split for us
    excl = pd.DataFrame([stats["pool"]], index=["pool"])
excl.loc["TOTAL"] = excl.sum()
print("\nExclusion report — REPORT THESE IN SECTION III-B:")
display(excl)

n_ab = int(excl.loc["TOTAL", "excluded_ab"])
if n_ab == 0:
    print("\n>>> No AB samples found. State: 'the dataset as collected contains no "
          "AB-reacting samples.'")
else:
    print(f"\n>>> WARNING: {n_ab} AB samples were previously mislabelled O-. "
          "All earlier results were on contaminated labels and must be re-run.")

dist = pd.DataFrame({
    "Train": pd.Series([s.label for s in train_s]).value_counts(),
    "Val":   pd.Series([s.label for s in val_s]).value_counts(),
    "Test":  pd.Series([s.label for s in test_s]).value_counts(),
}).reindex(range(NUM_CLASSES)).fillna(0).astype(int)
dist.index = BLOOD_GROUPS; dist["Total"] = dist.sum(axis=1)
print("\nClass distribution after safe AB exclusion:")
display(dist)
print(f"\nPooled dataset for cross-validation: {len(all_s)} samples")

Detected dataset layout:
  [train] images=/home/fawadsalamkhan/MyProjects/BloodGroup/train/images  (1860 files)
          labels=/home/fawadsalamkhan/MyProjects/BloodGroup/train/labels
  [val  ] images=/home/fawadsalamkhan/MyProjects/BloodGroup/valid/images  (178 files)
          labels=/home/fawadsalamkhan/MyProjects/BloodGroup/valid/labels
  [test ] images=/home/fawadsalamkhan/MyProjects/BloodGroup/test/images  (85 files)
          labels=/home/fawadsalamkhan/MyProjects/BloodGroup/test/labels
[train] total=1860  kept=1857  excluded_AB=  0  excluded_invalid=  3
[val  ] total= 178  kept= 178  excluded_AB=  0  excluded_invalid=  0
[test ] total=  85  kept=  85  excluded_AB=  0  excluded_invalid=  0

Exclusion report — REPORT THESE IN SECTION III-B:


,total,kept,excluded_ab,excluded_invalid
train,1860,1857,0,3
val,178,178,0,0
test,85,85,0,0
TOTAL,2123,2120,0,3



>>> No AB samples found. State: 'the dataset as collected contains no AB-reacting samples.'

Class distribution after safe AB exclusion:


,Train,Val,Test,Total
A+,0,0,0,0
A-,1857,178,85,2120
B+,0,0,0,0
B-,0,0,0,0
O+,0,0,0,0
O-,0,0,0,0



Pooled dataset for cross-validation: 2120 samples


## 15.2 Experiment 1 — fully-tuned baselines  *(R2 #2)*

⏱ The long one: 7 models × Optuna trials × seeds. Checkpoints after each model.

In [ ]:
df_base = exp1_baselines(train_s, val_s, test_s,
                         n_trials=N_TRIALS, tune_epochs=TUNE_EPOCHS,
                         final_epochs=FINAL_EPOCHS, n_seeds=N_SEEDS)
display(df_base)

In [ ]:
print(to_latex(df_base, drop_cols=["best_params"],
    caption="Comparison against independently and equivalently tuned baselines. Same "
            "Optuna budget, splits, sampler and schedule for all. Test accuracy is "
            "mean $\\pm$ std over seeds.",
    label="tab:baselines", path=f"{RESULTS_DIR}/table_baselines.tex"))

## 15.3 Experiment 2 — cross-validation  *(R2 #3)*

**Replaces the "100% on 24 samples" headline** with a cross-validated number + CI.

In [ ]:
best_hem = json.loads(
    df_base.loc[df_base.Model.str.contains("HemMaskNet"), "best_params"].iloc[0])
hp = HParams.from_optuna(best_hem, epochs=FINAL_EPOCHS)
print("HemMaskNet config used for CV:", hp)

df_cv, cv_summary = exp2_cross_validation(all_s, hp, backbone="efficientnet_b0",
                                          use_image=True, use_mask=True,
                                          n_splits=CV_SPLITS, n_repeats=CV_REPEATS)
display(df_cv)

In [ ]:
cm = np.array(cv_summary["oof_confusion_matrix"])
plt.figure(figsize=(6,5), dpi=150)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=BLOOD_GROUPS, yticklabels=BLOOD_GROUPS)
plt.xlabel("Predicted"); plt.ylabel("True")
plt.title(f"Out-of-fold confusion matrix (n={cv_summary['n_samples_oof']})")
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/fig_oof_confusion.png", bbox_inches="tight")
plt.show()
display(pd.read_csv(f"{RESULTS_DIR}/cv_per_class.csv"))

## 15.4 Experiment 3 — sensitivity  *(R2 #5)*

In [ ]:
set_seed(SEED)
final_model = DualStreamNet("efficientnet_b0", use_image=True, use_mask=True,
                            hidden1=hp.hidden1, hidden2=hp.hidden2,
                            dropout1=hp.dropout1, dropout2=hp.dropout2)
final_model, hist = train_model(final_model, train_s, val_s, hp, DEVICE, verbose=True)
torch.save(final_model.state_dict(), f"{RESULTS_DIR}/hemmasknet_final.pth")

df_sens = exp3_sensitivity(final_model, test_s, batch_size=hp.batch_size)
display(df_sens); plot_sensitivity(df_sens)

### 15.4b Does the mask stream make the model *brittle*?

Run the same corruptions on the image-only model. If HemMaskNet collapses when the
mask is wrong while the image-only model doesn't, that is a real finding — report it.

In [ ]:
best_img = json.loads(
    df_base.loc[df_base.Model.str.contains("EfficientNet-B0"), "best_params"].iloc[0])
hp_img = HParams.from_optuna(best_img, epochs=FINAL_EPOCHS)
set_seed(SEED)
img_model = DualStreamNet("efficientnet_b0", use_image=True, use_mask=False,
                          hidden1=hp_img.hidden1, hidden2=hp_img.hidden2,
                          dropout1=hp_img.dropout1, dropout2=hp_img.dropout2)
img_model, _ = train_model(img_model, train_s, val_s, hp_img, DEVICE)
df_sens_img = exp3_sensitivity(img_model, test_s, batch_size=hp_img.batch_size,
                               out_csv=f"{RESULTS_DIR}/table_sensitivity_imageonly.csv")
cmp = (df_sens[["Perturbation type","Setting","Accuracy (%)"]]
       .merge(df_sens_img[["Setting","Accuracy (%)"]], on="Setting",
              suffixes=(" HemMaskNet"," Image-only")))
cmp["Gap (pp)"] = (cmp["Accuracy (%) HemMaskNet"] - cmp["Accuracy (%) Image-only"]).round(2)
cmp.to_csv(f"{RESULTS_DIR}/table_sensitivity_comparison.csv", index=False)
display(cmp)

## 15.5 Experiment 4 — measured complexity & timing  *(R1 #3, R2 #6)*

In [ ]:
df_prof = exp4_profiling(hp, batch_sizes=(1,8,32), n_warmup=10, n_iters=100)
display(df_prof)
hem_cpu = df_prof[(df_prof.Model.str.contains("HemMaskNet")) &
                  (df_prof.Device=="CPU") & (df_prof.Batch==1)]
print("\nHeadline row for the low-cost-hardware claim:")
display(hem_cpu)
print(to_latex(df_prof,
    caption="Measured computational cost (mean over 100 forward passes after 10 warm-ups). "
            "Excludes the upstream YOLO detector.",
    label="tab:profiling", path=f"{RESULTS_DIR}/table_profiling.tex"))

> **State in the paper:** these numbers are the classifier only; the upstream YOLO
> detector that supplies the drop boxes is an additional deployment cost.

## 15.6 Experiment 5 — classical baseline  *(R2 #2)*

In [ ]:
df_classical = exp5_classical(train_s, test_s)
display(df_classical)

---
# Part 16 — Final summary: numbers for the paper

In [ ]:
summary = {
    "dataset": {"pooled_n": len(all_s),
                "excluded_AB": int(excl.loc["TOTAL","excluded_ab"]),
                "excluded_invalid": int(excl.loc["TOTAL","excluded_invalid"]),
                "train_n": len(train_s), "val_n": len(val_s), "test_n": len(test_s)},
    "cross_validation": {k:v for k,v in cv_summary.items() if k!="oof_confusion_matrix"},
    "smoke_test": SMOKE_TEST}
with open(f"{RESULTS_DIR}/final_summary.json","w") as f: json.dump(summary, f, indent=2)

cv = cv_summary
print("="*74)
print("NUMBERS TO PUT IN THE REVISED PAPER")
if SMOKE_TEST: print("!!! SMOKE TEST — NOT PUBLICATION-GRADE. Set SMOKE_TEST=False. !!!")
print("="*74)
print(f"AB samples excluded          : {summary['dataset']['excluded_AB']}")
print(f"Invalid samples excluded     : {summary['dataset']['excluded_invalid']}")
print(f"Pooled dataset size          : {len(all_s)}")
print(f"{CV_REPEATS}x{CV_SPLITS}-fold CV accuracy        : "
      f"{100*cv['fold_acc_mean']:.2f} +/- {100*cv['fold_acc_std']:.2f} %")
print(f"Out-of-fold accuracy (n={cv['n_samples_oof']}) : {100*cv['oof_accuracy']:.2f} % "
      f"[95% CI {100*cv['oof_accuracy_boot_ci'][0]:.1f}-{100*cv['oof_accuracy_boot_ci'][1]:.1f}]")
print(f"Out-of-fold macro-F1         : {cv['oof_macro_f1']:.3f} "
      f"[95% CI {cv['oof_macro_f1_boot_ci'][0]:.3f}-{cv['oof_macro_f1_boot_ci'][1]:.3f}]")
print(f"Out-of-fold ECE              : {cv['oof_ece']:.3f}")
print("\nBaselines (identical tuning budget):")
print(df_base[["Model","Test Acc (%)","Macro F1","Bootstrap 95% CI"]].to_string(index=False))
_w = df_sens.loc[df_sens["Accuracy (%)"].idxmin()]
print(f"\nWorst-case robustness        : {_w['Perturbation type']} / {_w['Setting']} "
      f"-> {_w['Accuracy (%)']:.1f}% ({_w['Δ vs clean (pp)']} pp vs clean)")
print("\nMeasured cost (HemMaskNet, CPU, batch 1):")
print(hem_cpu[["Params (M)","GFLOPs (bs=1)","Latency mean (ms)","Throughput (img/s)"]].to_string(index=False))
print(f"\nAll artefacts in ./{RESULTS_DIR}/  ->", sorted(os.listdir(RESULTS_DIR)))

---
# Part 17 — How to report this honestly

1. **Lead with the cross-validated out-of-fold accuracy + bootstrap CI**, not 100%/24.
2. Keep the single-split test result as a clearly-labelled secondary figure with its Wilson CI.
3. **Report the baseline table as it comes out.** If EfficientNet-only is within noise of
   HemMaskNet, say so — a reported inconvenient baseline is far more publishable than a hidden one.
4. If the dual model degrades faster than image-only under mask corruption (15.4b), report that too.
5. Seeds are fixed. Do not re-roll seeds until a number improves.
6. Cross-validation fixes the **statistical** objection, not external validity: one imaging
   context, one site, AB/weak-D still out of scope. Keep the multi-centre caveat.